# 第77课 · 听出一首歌的"调性指纹"——从零实现 chroma、RMS 能量与过零率（ZCR）

**目标**：实现三个音乐特有特征——chroma向量调性（tonality）、RMS能量包络（响度曲线）、节拍检测（BPM）——全部从 numpy 手动构建，无外部库。

🔗 **Aurora 连接**：`aurora.music` 子包已在 `src/aurora/music/features.py` 中完整实现，导出了 `chroma_vector`、`rms_envelope`、`zero_crossing_rate`、`onset_envelope` 和 `beat_track`。本节练习从零手写这些函数，就是为了搞懂它们内部到底怎么运作；完成后可与 `src/aurora/music/features.py` 的参考实现逐行对比。

← **上一课**　[L76 · 音乐理论速成](L76_music_theory.ipynb)

> 上节课学习了 **音乐理论速成**：音高、音程、色度轮与十二平均律。  
> 本课将探讨 **音乐特征工程**。

## 本课剧情：AI 怎么"听"出一首歌是什么调？

人类听到《月亮代表我的心》开头两个音，就知道是C大调。这个"直觉"背后是什么？

**调性感 = 哪些音高类别有最多的能量。**

C大调的曲子，C、E、G（主三和弦）的音频能量最大——把 FFT 功率谱按音高类别（chroma）分组加总，就能得到这个"调性指纹"：**chroma 向量，12维**。

**三个特征，三个问题**：

| 特征 | 回答什么问题 | 形状 |
|---|---|---|
| chroma_vector | "这段音乐在哪个调？用了什么和弦？" | (12,) |
| rms_envelope | "响度怎么随时间变化？哪里有强拍？" | (n_frames,) |
| zero_crossing_rate | "这是噪声（乐器过渡音）还是乐音？" | (n_frames,) |

**chroma 的计算逻辑**：
```
FFT功率谱 (n_fft//2+1,)
   ↓ 每个bin的频率 f = k × sr/n_fft
   ↓ 频率 → 音高类别 c = round(12×log2(f/440) + 9) % 12
   ↓ 按 c=0..11 把功率加总
   ↓ 归一化（除以总和或最大值）
   → chroma向量 (12,)
```

**RMS** 是"短时响度计"：每帧 `sqrt(mean(x^2))`，时间分辨率取决于 hop 大小。  
**ZCR** 数信号过零次数：乐音ZCR低（频率稳定），噪声ZCR高（随机过零）。

本节任务：从零实现这三个特征函数，用纯音和标准信号数值验证。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Chroma：调性指纹

STFT 给出每个 FFT bin 对应的频率 `f = k * sr / n_fft`（k = 0,1,...,n_fft//2）。把频率映射到 MIDI 音符：

```
midi = 12 * log2(f / 440) + 69
```

音高类（pitch class）= `round(midi) mod 12`，对应 C=0, C#=1, D=2, ..., B=11。把落在同一音高类的所有 bin 的功率加总，得到长度为 12 的向量，再归一化到 [0,1]。

直觉：A大调音阶（A B C# D E F# G#）演奏时，chroma 的第 0,2,4,6,9,11 位应显著高于其余。

In [ ]:
# 演示：构造一个只含 A4 (440 Hz) 的纯音，观察 chroma
sr = 22050
duration = 1.0
t = np.arange(int(sr * duration)) / sr
x_pure_a = np.sin(2 * np.pi * 440 * t)   # 纯 A4

n_fft = 2048
freqs = np.arange(n_fft // 2 + 1) * sr / n_fft  # 每个 bin 的频率

# 手动算功率谱（单帧演示）
frame = x_pure_a[:n_fft] * np.hanning(n_fft)
spectrum = np.abs(np.fft.rfft(frame)) ** 2

# 音高类映射
with np.errstate(divide='ignore', invalid='ignore'):
    midi = np.where(freqs > 0, 12 * np.log2(freqs / 440) + 69, -1)
pitch_class = (np.round(midi).astype(int)) % 12

chroma_demo = np.zeros(12)
for pc, p in zip(pitch_class, spectrum):
    if 0 <= pc < 12:
        chroma_demo[pc] += p
chroma_demo /= (chroma_demo.max() + 1e-8)

labels = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
plt.figure(figsize=(8, 3))
plt.bar(labels, chroma_demo)
plt.title('Chroma — 纯 A4 (440 Hz)；期望 A=9 最高')
plt.ylabel('归一化能量')
plt.tight_layout()
plt.show()
print(f'能量最高的音高类 index = {chroma_demo.argmax()}  (应为 9 = A)')

## 2. RMS 能量包络

对音频信号做短时分帧（帧长 `frame_len`，跳步 `hop`），对每帧计算均方根：

```
rms[i] = sqrt( mean( frame_i ** 2 ) )
```

结果是长度为 `n_frames` 的向量，描述响度随时间的变化。静音段 RMS ≈ 0，击鼓/齐奏段 RMS 可达 0.3–0.8（归一化到 ±1 的音频）。

帧数公式：`n_frames = 1 + (len(x) - frame_len) // hop`。

In [ ]:
# 演示：方波 + 静音拼接，观察 RMS 包络
sr = 22050
silence = np.zeros(sr // 2)            # 0.5 s 静音
loud = 0.8 * np.sign(np.sin(2 * np.pi * 220 * np.arange(sr) / sr))  # 方波 1s
x_demo = np.concatenate([silence, loud, silence])

frame_len, hop = 2048, 512
n_frames = 1 + (len(x_demo) - frame_len) // hop
frames = np.array([x_demo[i*hop : i*hop + frame_len] for i in range(n_frames)])
rms_demo = np.sqrt(np.mean(frames ** 2, axis=1))

times = np.arange(n_frames) * hop / sr
plt.figure(figsize=(9, 3))
plt.plot(times, rms_demo)
plt.xlabel('时间 (s)')
plt.ylabel('RMS')
plt.title('RMS 能量包络：静音→方波→静音')
plt.tight_layout()
plt.show()

## 3. 节拍检测（自相关（autocorrelation）法）

**思路**：
1. 计算 RMS 能量包络（onset strength 的粗略替代）。
2. 对包络做自相关：`ac[lag] = sum( rms[t] * rms[t+lag] )`。
3. 在对应 BPM 范围（通常 60–240 BPM）的 lag 区间内找峰值 lag*。
4. `BPM = 60 / (lag* * hop / sr)`。

```
lag_min = int(60 / 240 * sr / hop)   # 对应 240 BPM
lag_max = int(60 / 60  * sr / hop)   # 对应  60 BPM
lag_star = lag_min + ac[lag_min:lag_max].argmax()
BPM = 60 / (lag_star * hop / sr)
```

只要出现周期性的节拍，自相关就会在对应的 lag 上冒出明显的峰。这种办法不用标注数据，属于无监督估计。

In [ ]:
# 演示：构造 120 BPM 的击鼓包络，验证自相关 BPM 估计
sr = 22050
hop = 512
frame_len = 2048
bpm_true = 120
beat_period_samples = int(60 / bpm_true * sr)   # 每拍的采样数

# 用衰减脉冲模拟击鼓瞬态
x_drum = np.zeros(sr * 4)   # 4 s
for beat_start in range(0, len(x_drum), beat_period_samples):
    end = min(beat_start + 2205, len(x_drum))  # 100 ms 衰减
    length = end - beat_start
    x_drum[beat_start:end] += np.exp(-np.arange(length) / 1000) * np.random.randn(length) * 0.5

# RMS 包络
n_frames = 1 + (len(x_drum) - frame_len) // hop
frames_d = np.array([x_drum[i*hop:i*hop+frame_len] for i in range(n_frames)])
rms_d = np.sqrt(np.mean(frames_d**2, axis=1))

# 自相关
ac = np.correlate(rms_d, rms_d, mode='full')
ac = ac[len(ac)//2:]   # 只要正 lag

lag_min = max(1, int(60/240 * sr / hop))
lag_max = int(60/60  * sr / hop)
lag_star = lag_min + ac[lag_min:lag_max].argmax()
bpm_est = 60 / (lag_star * hop / sr)

print(f'真实 BPM = {bpm_true},  估计 BPM = {bpm_est:.1f}')

plt.figure(figsize=(9,3))
lags = np.arange(lag_min, lag_max) * hop / sr
plt.plot(lags, ac[lag_min:lag_max])
plt.axvline(lag_star * hop / sr, color='r', linestyle='--', label=f'峰值 lag → {bpm_est:.1f} BPM')
plt.xlabel('Lag (s)')
plt.ylabel('自相关')
plt.title('RMS 包络自相关 — 峰值对应节拍周期')
plt.legend()
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `chroma_vector(power_spectrum, sr, n_fft)`

**签名**：
```python
def chroma_vector(power_spectrum: np.ndarray, sr: int, n_fft: int) -> np.ndarray:
    # power_spectrum: (n_fft//2+1,) 功率谱（幅度的平方）
    # 返回: (12,) float, chroma向量，已归一化（除以总和，若全零则全零）
```

**4步实现路线**：

| 步骤 | 操作 | 公式 |
|---|---|---|
| 1 | 计算每个FFT bin的频率 | `f[k] = k × sr / n_fft`，k=0..n_fft//2 |
| 2 | 频率→音高类别（跳过f=0，取对数无意义） | `c = round(12×log2(f/440) + 69) % 12` |
| 3 | 按chroma分组加总功率 | `chroma[c] += power_spectrum[k]` |
| 4 | 归一化 | `chroma /= chroma.sum()`（若sum==0则保持全零） |

> 常数 **69** 是 A4=440Hz 的 MIDI 编号（69 = 5×12 + 9），mod 12 后与直接写 `+9` 等价——本课统一用 +69，与引言公式和参考实现一致。

**验收标准**：
- 纯A4（440Hz）信号：`chroma_vector(...)[9]` 为最大值（A=chroma 9）
- 输出shape=(12,)，值全非负，sum≈1.0（或全零）

In [ ]:
def chroma_vector(power_spectrum, sr, n_fft):
    """
    将功率谱映射到 12 维 chroma 向量。

    Parameters
    ----------
    power_spectrum : np.ndarray, shape (n_fft//2 + 1,)
    sr : int  采样率
    n_fft : int  FFT 点数

    Returns
    -------
    chroma : np.ndarray, shape (12,)  归一化（除以总和，若全零则全零）
    """
    raise NotImplementedError("chroma_vector: 请完成实现 — 参考4步实现路线")
    n_bins = len(power_spectrum)
    freqs = np.arange(n_bins) * sr / n_fft  # ✏️ TODO: 每个 bin 的频率

    chroma = np.zeros(12)
    for k, (f, p) in enumerate(zip(freqs, power_spectrum)):
        if f <= 0:
            continue
        # ✏️ TODO: 计算 midi = 12 * log2(f/440) + 69
        midi = 0.0
        # ✏️ TODO: pc = round(midi) mod 12，累加到 chroma[pc]
        pass

    # ✏️ TODO: 归一化（除以总和，若sum==0则全零）
    return chroma

In [ ]:
# 检查：纯 A4 信号的 chroma[9] 应最高
try:
    sr_c, n_fft_c = 22050, 2048
    t_c = np.arange(n_fft_c) / sr_c
    x_c = np.sin(2 * np.pi * 440 * t_c) * np.hanning(n_fft_c)
    pspec = np.abs(np.fft.rfft(x_c)) ** 2

    ch = chroma_vector(pspec, sr_c, n_fft_c)
    peak_pc = ch.argmax()
    assert peak_pc == 9, f'期望 9 (A), 得到 {peak_pc}'
    assert abs(ch.sum() - 1.0) < 1e-6, f'应 L1 归一化（除以总和），sum={ch.sum():.6f}'
    assert ch[9] > 0.99, f'A4 纯音能量应集中在 chroma[9]，得 {ch[9]:.4f}'
    print(f'✅ chroma_vector OK：peak pitch class = {peak_pc} (A)，chroma[9] = {ch[9]:.4f}，sum = {ch.sum():.4f}')
except (NotImplementedError, TypeError, AttributeError) as e:
    print(f'⏸ chroma_vector 尚未实现：{e}\n请按4步实现路线完成上方 TODO。')

## 5. ✏️ 实现 `rms_envelope(x, frame_len=2048, hop=512)`

**签名**：
```python
def rms_envelope(x: np.ndarray, frame_len: int = 2048, hop: int = 512) -> np.ndarray:
    # x: 1D音频数组
    # 返回: (n_frames,) float，逐帧RMS值
```

**3步实现**：

| 步骤 | 操作 | 细节 |
|---|---|---|
| 1 | 分帧：`n_frames = 1 + (len(x) - frame_len) // hop` | 第i帧起点: `i * hop` |
| 2 | 对每帧计算RMS | `sqrt(mean(frame^2))` |
| 3 | 返回 shape=(n_frames,) | 可用列表append或预分配数组 |

**验收标准**：
- 全零信号 → RMS全为0.0
- 全1信号（长度=frame_len）→ RMS=1.0
- 时域帧数公式：`n_frames = 1 + (N - frame_len) // hop`

In [ ]:
def rms_envelope(x, frame_len=2048, hop=512):
    """
    计算音频信号的逐帧 RMS 能量包络。

    Parameters
    ----------
    x : np.ndarray, shape (N,)  单声道音频
    frame_len : int  帧长（采样点数）
    hop : int  跳步

    Returns
    -------
    rms : np.ndarray, shape (n_frames,)
    """
    # ✏️ TODO: 计算 n_frames = 1 + (len(x) - frame_len) // hop
    # ✏️ TODO: 切帧 frames[i] = x[i*hop : i*hop+frame_len]，shape = (n_frames, frame_len)
    # ✏️ TODO: 返回每帧的 sqrt(mean(frame**2))
    raise NotImplementedError("rms_envelope: 请完成实现 — 参考推理路线三步")

In [ ]:
# 检查 rms_envelope
try:
    # 静音 → RMS ≈ 0
    x_sil = np.zeros(10000)
    rms_sil = rms_envelope(x_sil)
    assert rms_sil.max() < 1e-10, f'静音 RMS 应约为 0，得 {rms_sil.max()}'

    # 常数信号 0.5 → RMS ≈ 0.5
    x_const = np.ones(10000) * 0.5
    rms_const = rms_envelope(x_const)
    assert np.allclose(rms_const, 0.5, atol=1e-6), f'常数信号 RMS 应为 0.5，得 {rms_const.mean():.4f}'

    print(f'✅ rms_envelope OK：静音 max={rms_sil.max():.2e}，常数 mean={rms_const.mean():.4f}')
except (NotImplementedError, TypeError, AttributeError) as e:
    print(f'⏸ rms_envelope 尚未实现：{e}\n请按推理路线完成 cell 13 中的 TODO。')

## 6. 参数实验：hop 对 RMS 时间分辨率的影响

改变 `hop` 参数，观察 RMS 包络的时间分辨率与平滑程度：

- `hop=128`：高时间分辨率，包络抖动，能捕捉到快速瞬态（< 6 ms）
- `hop=512`（默认）：均衡，约 23 ms 步长，适合节拍检测
- `hop=2048`：低分辨率，包络极平滑，接近整体响度，节拍细节丢失

预期现象：`hop` 越大，曲线越平滑，节拍峰越模糊；`hop` 越小，曲线越细腻，但计算量正比增大。

In [ ]:
# 参数实验：不同 hop 的 RMS 包络对比
sr_exp = 22050
bpm_exp = 120
beat_period = int(60 / bpm_exp * sr_exp)
x_exp = np.zeros(sr_exp * 4)
for bs in range(0, len(x_exp), beat_period):
    end = min(bs + 2205, len(x_exp))
    length = end - bs
    x_exp[bs:end] += np.exp(-np.arange(length) / 800) * 0.9

frame_len_exp = 2048
hops = [128, 512, 2048]
colors = ['tab:blue', 'tab:orange', 'tab:green']

try:
    rms_by_hop = [(h, c, rms_envelope(x_exp, frame_len=frame_len_exp, hop=h))
                  for h, c in zip(hops, colors)]
except (NotImplementedError, TypeError):
    print('⏸ rms_envelope 尚未实现，跳过参数实验；完成上方 TODO 后重新运行本格。')
else:
    plt.figure(figsize=(11, 4))
    for h, c, rms_e in rms_by_hop:
        t_e = np.arange(len(rms_e)) * h / sr_exp
        plt.plot(t_e, rms_e, label=f'hop={h}', color=c, alpha=0.8)

    plt.xlabel('时间 (s)')
    plt.ylabel('RMS')
    plt.title('不同 hop 的 RMS 包络（120 BPM 模拟击鼓）')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print('观察：hop=128 最细腻，hop=2048 最平滑；节拍峰在小 hop 时更清晰。')

## 7. ✏️ 实现 `zero_crossing_rate(x, frame_len=2048, hop=512)`

**签名**：
```python
def zero_crossing_rate(x: np.ndarray, frame_len: int = 2048, hop: int = 512) -> np.ndarray:
    # 返回: (n_frames,) float，每帧的过零率（0-1之间）
```

**过零率定义**：
```
ZCR[i] = (1 / frame_len) × #{n: sign(x[n]) ≠ sign(x[n+1]), n ∈ 帧i}
```

**实现**：
```python
# 对每帧：
zcr = np.mean(np.abs(np.diff(np.sign(frame))) / 2)
# np.sign(frame): 每个样本的符号（-1, 0, 1）
# np.diff(...): 相邻符号差；非零=过零
# /2: 归一化到[0,1]
```

**验收标准**：
- 纯音（440Hz正弦波，22050Hz采样）：ZCR ≈ 2×440/22050 ≈ 0.04
- 全零信号：ZCR=0.0
- 白噪声（随机符号）：ZCR接近0.5

In [ ]:
def zero_crossing_rate(x, frame_len=2048, hop=512):
    """
    计算音频信号的逐帧过零率（Zero Crossing Rate）。

    Parameters
    ----------
    x : np.ndarray, shape (N,)
    frame_len : int
    hop : int

    Returns
    -------
    zcr : np.ndarray, shape (n_frames,)  每帧过零率，范围 [0, 0.5]
    """
    # ✏️ TODO: 1. 计算 n_frames（与 rms_envelope 相同公式）
    # ✏️ TODO: 2. 切帧
    # ✏️ TODO: 3. 对每帧 np.diff(np.sign(frame)) 统计非零数 / (2 * frame_len)
    raise NotImplementedError("zero_crossing_rate: 请完成实现")


In [ ]:
# 检查 zero_crossing_rate
try:
    sr_z = 22050
    # 纯 A4 正弦波：ZCR ≈ 2 * 440 / 22050 ≈ 0.0399
    t_z = np.arange(sr_z) / sr_z
    x_sine = np.sin(2 * np.pi * 440 * t_z)
    zcr_sine = zero_crossing_rate(x_sine, frame_len=2048, hop=512)
    expected_zcr = 2 * 440 / sr_z   # ≈ 0.0399
    assert np.allclose(zcr_sine.mean(), expected_zcr, atol=0.005), \
        f'正弦 ZCR 应约为 {expected_zcr:.4f}，得 {zcr_sine.mean():.4f}'

    # 白噪声：ZCR ≈ 0.5（随机正负号，过零极频繁）
    rng = np.random.default_rng(42)
    x_noise = rng.standard_normal(sr_z)
    zcr_noise = zero_crossing_rate(x_noise, frame_len=2048, hop=512)
    assert zcr_noise.mean() > 0.45, f'白噪声 ZCR 应 > 0.45，得 {zcr_noise.mean():.4f}'

    print(f'✅ zero_crossing_rate OK：正弦 ZCR mean={zcr_sine.mean():.4f}（期望≈{expected_zcr:.4f}），'
          f'噪声 ZCR mean={zcr_noise.mean():.4f}（期望≈0.5）')
except (NotImplementedError, TypeError, AttributeError) as e:
    print(f'⏸ zero_crossing_rate 尚未实现：{e}\n请按推理路线完成上方 stub。')

## 本课收束

本节实现了 `chroma_vector`（12维调性指纹，输出归一化 ndarray）和 `rms_envelope`（逐帧均方根，输出响度时间序列），并演示了自相关节拍估计（BPM 定位）——三者将共同构成 `aurora.music` 特征提取管线的核心。下一节（**L78**）将从零实现节拍追踪：计算 onset 包络、用自相关估计 BPM，并在合成鼓点信号上验证精度。

## ✏️ 白板挑战：音乐特征手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：功率谱中 k=5 对应的频率是多少？（sr=22050, n_fft=2048）  
（f=k×sr/n_fft=5×22050/2048≈53.8 Hz）

**问 2**：440 Hz（A4）对应的 chroma 值是几？推导过程？  
（c=round(12×log2(440/440)+69)%12=69%12=9；A=chroma 9。69=5×12+9，是 A4 的 MIDI 编号）

**问 3**：若 sr=22050，frame_len=2048，hop=512，信号长度 N=44100，有多少帧？  
（n_frames=1+(44100-2048)//512=1+42052//512=1+82=83帧）

**问 4**：为什么纯音的 ZCR ≈ 2×f/sr？  
（频率为f的正弦波每秒过零2f次（每个周期2次），除以采样率sr得每样本的过零率）

**问 5**：chroma_vector 归一化为什么用"除以总和"而不是"除以最大值"？  
（总和归一化使chroma向量代表"能量分布概率"，可直接计算两个chroma向量的余弦相似度来比较调性；最大值归一化会丢失各分量之间的相对强度信息）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

sr_test = 22050
n_fft_test = 2048
hop_test = 512
frame_len_test = 2048

# 问1：FFT bin频率
k5_freq = 5 * sr_test / n_fft_test
expected_k5 = 53.8  # 近似
assert abs(k5_freq - expected_k5) < 0.2, f"k=5频率={k5_freq:.2f}，期望≈{expected_k5}"
print(f"Q1 ✅  k=5: f={k5_freq:.2f} Hz（≈{expected_k5} Hz）")

# 问2：A4的chroma
import math
freq_a4 = 440.0
c_a4 = round(12 * math.log2(freq_a4 / 440) + 69) % 12
assert c_a4 == 9, f"440Hz→chroma应=9，得{c_a4}"
print(f"Q2 ✅  440Hz(A4)→chroma=round(12×log2(1)+69)%12=69%12={c_a4}（A=9，69=5×12+9）")

# 问3：帧数公式
N_sig = 44100
n_frames_calc = 1 + (N_sig - frame_len_test) // hop_test
assert n_frames_calc == 83, f"帧数应=83，得{n_frames_calc}"
print(f"Q3 ✅  N={N_sig},frame={frame_len_test},hop={hop_test}→n_frames=1+({N_sig}-{frame_len_test})//{hop_test}={n_frames_calc}")

# 问4：ZCR验证
try:
    t_zcr = np.arange(frame_len_test) / sr_test
    f_zcr = 440.0
    x_sine = np.sin(2 * np.pi * f_zcr * t_zcr)
    zcr_val = zero_crossing_rate(x_sine, frame_len=frame_len_test, hop=frame_len_test)
    expected_zcr = 2 * f_zcr / sr_test
    assert abs(zcr_val[0] - expected_zcr) < 0.005, f"ZCR≈{expected_zcr:.4f}，得{zcr_val[0]:.4f}"
    print(f"Q4 ✅  纯音{f_zcr}Hz: ZCR={zcr_val[0]:.4f}≈2×{f_zcr}/{sr_test}={expected_zcr:.4f}")
except (NotImplementedError, TypeError):
    expected_zcr = 2 * 440 / sr_test
    print(f"Q4 ✅  ZCR理论：2×f/sr=2×440/{sr_test}≈{expected_zcr:.4f}（待实现后验证）")

# 问5：chroma归一化（验证总和归一化的性质）
try:
    t_chk = np.arange(n_fft_test) / sr_test
    x_a4 = np.sin(2 * np.pi * 440 * t_chk)
    ps_a4 = np.abs(np.fft.rfft(x_a4)) ** 2
    ch = chroma_vector(ps_a4, sr_test, n_fft_test)
    assert abs(ch.sum() - 1.0) < 1e-6 or ch.sum() == 0, f"chroma总和应≈1.0，得{ch.sum():.6f}"
    assert ch[9] == ch.max(), f"A4的chroma[9]应最大，实际max位置={ch.argmax()}"
    print(f"Q5 ✅  chroma总和归一化：sum={ch.sum():.4f}，最大值在chroma[{ch.argmax()}]=A={ch[9]:.4f}")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  总和归一化→概率分布，余弦相似度可直接比较调性（待实现后验证）")

print("\n🎉 音乐特征白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l77_review = {
    "chroma_logic_understood":  None,  # 理解chroma=FFT功率按音高类别分组加总？True/False
    "chroma_vector_impl":       None,  # chroma_vector()实现正确，A4纯音→chroma[9]最大？True/False
    "rms_envelope_impl":        None,  # rms_envelope()实现正确，全零→0，全1→1？True/False
    "zcr_impl":                 None,  # zero_crossing_rate()实现正确，ZCR≈2f/sr？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l77_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l77_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L77 全部通关！进入 L78：节拍追踪从零实现')

---

→ **下一课**　[L78 · 节拍追踪从零实现](L78_beat_tracking.ipynb)

> 下节课将学习 **节拍追踪从零实现**：onset 包络、自相关与 BPM 估计。